# MV Validation — Group-Aware Train / Validation / Test Split

## Objective

This notebook independently reads the permanently saved feature table used by the LightGBM notebook and creates a new `label_split` under one hard constraint:

> All rows belonging to the same `login_id + acct_nbr` pair must stay in exactly one split.

The new split also tries to remain close to the Model Development split in terms of:

1. total row count;
2. positive-label count;
3. positive rate;
4. number of rows moved away from their original split.

The notebook does **not** retrain the model and does **not** overwrite the original feature table. After validation, it writes a new complete Delta feature table with the repaired split.

## Source confirmed from the LightGBM notebook

The original LightGBM notebook reads:

```text
insider_us_nms_features_table_v2
```

and defines:

```text
label_split = train / val / test
```

The current split is therefore recovered directly from the saved feature table rather than reconstructed from temporary notebook variables.


In [ ]:
# Core imports

import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.window import Window

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 1. Configuration

Only the saved feature table is required.

The weights below define how strongly the allocation algorithm prioritizes each target:

- `ROW_WEIGHT`: closeness to the original row-count allocation;
- `POSITIVE_WEIGHT`: closeness to the original positive-label allocation;
- `MOVE_WEIGHT`: penalty for rows moved away from their original split;
- `OVERSHOOT_WEIGHT`: additional penalty when a split exceeds its target.

Positive balance receives the largest ordinary weight because positive observations are scarce and directly affect model training and recall evaluation.


In [ ]:
# Saved feature table used by the LightGBM notebook
FEATURE_TABLE_PATH = (
    "abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/"
    "ins_us_nms/v1/output/insider_us_nms_features_table_v2"
)

# Column names
EMPLOYEE_COL = "login_id"
ACCOUNT_COL = "acct_nbr"
SAMPLE_DATE_COL = "date"
WINDOW_START_COL = "lookback_window_start"
WINDOW_END_COL = "fraud_date"
TARGET_COL = "label"
ORIGINAL_SPLIT_COL = "label_split"
NEW_SPLIT_COL = "new_label_split"

# Expected split labels from the LightGBM notebook
SPLIT_ORDER = ["train", "val", "test"]

# Reproducibility
RANDOM_SEED = 42

# Objective-function weights
ROW_WEIGHT = 1.0
POSITIVE_WEIGHT = 3.0
MOVE_WEIGHT = 0.5
OVERSHOOT_WEIGHT = 5.0

# Number of randomized greedy attempts.
# More attempts may improve the final allocation but increase runtime.
N_ATTEMPTS = 10

# Small number used to avoid division by zero.
EPSILON = 1e-12

# Rebuild cv_fold for the repaired training split.
N_CV_FOLDS = 5
CV_FOLD_SEED = 43

# New complete feature table; the original feature table is not overwritten.
FULL_FEATURE_OUTPUT_PATH = (
    'abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/'
    'mrm/ins_us_nms/output/insider_us_nms_features_table_group_aware_split_v1'
)

## 2. Read the permanently saved feature table

Only columns required for the split analysis are selected.

No model features are needed because this notebook is analyzing and rebuilding `label_split`, not generating predictions.


In [ ]:
features_df = (
    spark.read
    .format("delta")
    .load(FEATURE_TABLE_PATH)
    .select(
        F.upper(F.col(EMPLOYEE_COL)).alias(EMPLOYEE_COL),
        F.col(ACCOUNT_COL).cast('string').alias(ACCOUNT_COL),
        F.to_date(SAMPLE_DATE_COL).alias(SAMPLE_DATE_COL),
        F.to_date(WINDOW_START_COL).alias(WINDOW_START_COL),
        F.to_date(WINDOW_END_COL).alias(WINDOW_END_COL),
        F.col(TARGET_COL).cast('int').alias(TARGET_COL),
        F.col(ORIGINAL_SPLIT_COL),
    )
    .cache()
)

invalid_input_count = features_df.filter(
    F.col(EMPLOYEE_COL).isNull()
    | F.col(ACCOUNT_COL).isNull()
    | F.col(SAMPLE_DATE_COL).isNull()
    | F.col(WINDOW_START_COL).isNull()
    | F.col(WINDOW_END_COL).isNull()
    | (F.col(WINDOW_START_COL) > F.col(WINDOW_END_COL))
    | ~F.col(TARGET_COL).isin(0, 1)
    | ~F.col(ORIGINAL_SPLIT_COL).isin(SPLIT_ORDER)
).count()
assert invalid_input_count == 0, f'Found {invalid_input_count} invalid input rows'

display(features_df.limit(10))

print("Total rows:", features_df.count())

## 3. Recover the original MD split distribution

This table establishes the target allocation.

For each original split, the notebook calculates:

- number of rows;
- row proportion;
- number of positive rows;
- share of all positive rows;
- number of negative rows;
- positive rate;
- number of distinct employee-account pairs.

These original proportions become the targets for the new group-aware split.


In [ ]:
total_rows = features_df.count()
total_positives = (
    features_df
    .agg(F.sum(TARGET_COL).alias("total_positives"))
    .first()["total_positives"]
)
total_negatives = total_rows - total_positives

original_split_summary = (
    features_df
    .groupBy(ORIGINAL_SPLIT_COL)
    .agg(
        F.count("*").alias("row_count"),
        F.sum(TARGET_COL).alias("positive_count"),
        (
            F.count("*") - F.sum(TARGET_COL)
        ).alias("negative_count"),
        F.avg(TARGET_COL).alias("positive_rate"),
        F.countDistinct(
            F.struct(EMPLOYEE_COL, ACCOUNT_COL)
        ).alias("pair_count")
    )
    .withColumn(
        "row_share",
        F.col("row_count") / F.lit(total_rows)
    )
    .withColumn(
        "positive_share",
        F.col("positive_count") / F.lit(total_positives)
    )
    .orderBy(
        F.when(F.col(ORIGINAL_SPLIT_COL) == "train", 1)
         .when(F.col(ORIGINAL_SPLIT_COL) == "val", 2)
         .when(F.col(ORIGINAL_SPLIT_COL) == "test", 3)
         .otherwise(4)
    )
)

display(original_split_summary)

## 4. Check the current employee-account leakage

A pair is considered cross-split when the same `login_id + acct_nbr` appears in more than one of:

```text
train / val / test
```

This check quantifies the current issue before creating a replacement split.


In [ ]:
pair_current_split_check = (
    features_df
    .groupBy(EMPLOYEE_COL, ACCOUNT_COL)
    .agg(
        F.count("*").alias("row_count"),
        F.sum(TARGET_COL).alias("positive_count"),
        F.countDistinct(
            ORIGINAL_SPLIT_COL
        ).alias("number_of_splits"),
        F.collect_set(
            ORIGINAL_SPLIT_COL
        ).alias("splits")
    )
)

current_leakage_summary = (
    pair_current_split_check
    .agg(
        F.count("*").alias("total_pairs"),
        F.sum(
            F.when(
                F.col("number_of_splits") > 1,
                1
            ).otherwise(0)
        ).alias("pairs_crossing_splits"),
        F.sum(
            F.when(
                F.col("number_of_splits") > 1,
                F.col("row_count")
            ).otherwise(0)
        ).alias("rows_in_crossing_pairs"),
        F.sum(
            F.when(
                F.col("number_of_splits") > 1,
                F.col("positive_count")
            ).otherwise(0)
        ).alias("positives_in_crossing_pairs")
    )
    .withColumn(
        "crossing_pair_rate",
        F.col("pairs_crossing_splits") / F.col("total_pairs")
    )
    .withColumn(
        "crossing_row_rate",
        F.col("rows_in_crossing_pairs") / F.lit(total_rows)
    )
    .withColumn(
        "crossing_positive_rate",
        F.col("positives_in_crossing_pairs") / F.lit(total_positives)
    )
)

display(current_leakage_summary)

## 5. Separate fixed and crossing employee-account pairs

Each pair is one indivisible allocation unit. Pairs already contained in one original split remain fixed. Only pairs currently crossing splits are sent to the greedy repair algorithm.

For every pair, the notebook calculates:

- `row_count`;
- `positive_count`;
- `negative_count`;
- original row count in each split.

Once a crossing pair is repaired, every row from that pair receives the same new split.


In [ ]:
pair_summary_spk = (
    features_df
    .groupBy(EMPLOYEE_COL, ACCOUNT_COL)
    .agg(
        F.count("*").alias("row_count"),
        F.sum(TARGET_COL).alias("positive_count")
    )
    .withColumn(
        "negative_count",
        F.col("row_count") - F.col("positive_count")
    )
)

print("Number of employee-account pairs:", pair_summary_spk.count())

crossing_pair_ids = (
    pair_current_split_check
    .filter(F.col('number_of_splits') > 1)
    .select(EMPLOYEE_COL, ACCOUNT_COL)
    .cache()
)

# Non-crossing pairs keep their original split exactly.
non_crossing_assignment_spk = (
    features_df
    .join(crossing_pair_ids, on=[EMPLOYEE_COL, ACCOUNT_COL], how='left_anti')
    .select(
        EMPLOYEE_COL, ACCOUNT_COL,
        F.col(ORIGINAL_SPLIT_COL).alias(NEW_SPLIT_COL),
    )
    .distinct()
)

fixed_summary_rows = (
    features_df
    .join(crossing_pair_ids, on=[EMPLOYEE_COL, ACCOUNT_COL], how='left_anti')
    .groupBy(ORIGINAL_SPLIT_COL)
    .agg(
        F.count('*').alias('rows'),
        F.sum(TARGET_COL).alias('positives'),
    )
    .collect()
)
fixed_totals = {split: {'rows': 0.0, 'positives': 0.0} for split in SPLIT_ORDER}
for row in fixed_summary_rows:
    fixed_totals[row[ORIGINAL_SPLIT_COL]] = {
        'rows': float(row['rows']),
        'positives': float(row['positives']),
    }

crossing_pair_summary_spk = (
    features_df
    .join(crossing_pair_ids, on=[EMPLOYEE_COL, ACCOUNT_COL], how='inner')
    .groupBy(EMPLOYEE_COL, ACCOUNT_COL)
    .agg(
        F.count('*').alias('row_count'),
        F.sum(TARGET_COL).alias('positive_count'),
        *[
            F.sum(
                F.when(F.col(ORIGINAL_SPLIT_COL) == split, 1).otherwise(0)
            ).alias(f'original_{split}_rows')
            for split in SPLIT_ORDER
        ],
    )
)
crossing_pair_pd = crossing_pair_summary_spk.toPandas()

integer_cols = ['row_count', 'positive_count'] + [
    f'original_{split}_rows' for split in SPLIT_ORDER
]
for column in integer_cols:
    crossing_pair_pd[column] = crossing_pair_pd[column].astype(np.int64)

total_crossing_rows = int(crossing_pair_pd['row_count'].sum())
print('Crossing pairs to repair:', len(crossing_pair_pd))
print('Rows in crossing pairs:', total_crossing_rows)
display(crossing_pair_pd.head(10))

## 6. Build the allocation targets from the original split

The new split should remain as close as possible to the original MD split.

For each split, targets are based on the original:

- share of total rows;
- share of all positive rows;
- positive rate.

Using the original shares is preferable to manually assuming 70/15/15 because it reproduces the actual saved MD allocation.


In [ ]:
original_target_rows = (
    original_split_summary
    .filter(F.col(ORIGINAL_SPLIT_COL).isin(SPLIT_ORDER))
    .select(
        ORIGINAL_SPLIT_COL, 'row_count', 'row_share',
        'positive_count', 'positive_share', 'positive_rate',
    )
    .collect()
)
targets = {
    row[ORIGINAL_SPLIT_COL]: {
        'rows': int(row['row_count']),
        'row_share': float(row['row_share']),
        'positives': int(row['positive_count']),
        'positive_share': float(row['positive_share']),
        'positive_rate': float(row['positive_rate']),
    }
    for row in original_target_rows
}
assert set(targets) == set(SPLIT_ORDER), f'Unexpected target splits: {targets}'

print("Allocation targets:")
for split in SPLIT_ORDER:
    print(split, targets[split])

## 7. Objective function

For each candidate split, the algorithm estimates the complete train/validation/test result after assigning the current crossing pair.

The candidate score is:

```text
loss =
    ROW_WEIGHT       × Σ normalized row error
  + POSITIVE_WEIGHT  × Σ normalized positive-count error
  + MOVE_WEIGHT      × moved rows / crossing rows
  + OVERSHOOT_WEIGHT × Σ normalized overshoot
```

Where:

```text
normalized row error
= |projected rows - target rows| / target rows
```

The sums cover all three splits. Moving a complete pair to split `s` costs the number of that pair's rows that originally belonged to the other two splits.

The overshoot penalty discourages continuing to add pairs to a split that has already exceeded its target.

The pair is assigned to the split with the lowest global loss.

This is a greedy approximation, not a mathematical proof of the global optimum. Multiple randomized attempts are used, and the allocation with the lowest final objective is retained.


In [ ]:
def normalized_absolute_error(actual, target):
    return abs(actual - target) / max(target, EPSILON)


def normalized_overshoot(actual, target):
    return max(actual - target, 0.0) / max(target, EPSILON)


def allocation_objective(current_totals, moved_rows):
    row_error = 0.0
    positive_error = 0.0
    overshoot_error = 0.0

    for split in SPLIT_ORDER:
        target = targets[split]
        actual = current_totals[split]
        row_error += normalized_absolute_error(actual['rows'], target['rows'])
        positive_error += normalized_absolute_error(actual['positives'], target['positives'])
        overshoot_error += normalized_overshoot(actual['rows'], target['rows'])
        overshoot_error += normalized_overshoot(actual['positives'], target['positives'])

    move_error = moved_rows / max(total_crossing_rows, 1)
    return (
        ROW_WEIGHT * row_error
        + POSITIVE_WEIGHT * positive_error
        + MOVE_WEIGHT * move_error
        + OVERSHOOT_WEIGHT * overshoot_error
    )

## 8. Minimal-change balanced greedy repair

Important implementation details:

1. Non-crossing pairs remain in their original split.
2. Only crossing pairs are allocated, once per complete pair.
3. Pairs are ordered by row count with randomized tie-breaking; positive count is not used for ordering.
4. Every candidate is scored using the global state of train, validation, and test.
5. Several attempts are run, and the allocation with the lowest final loss is kept.


In [ ]:
def allocate_crossing_pairs(crossing_pairs, random_seed):
    work = crossing_pairs.copy()
    rng = np.random.default_rng(random_seed)
    work['_tie_breaker'] = rng.random(len(work))

    # Large pairs are harder to place, but label is never used for ordering.
    work = work.sort_values(
        ['row_count', '_tie_breaker'],
        ascending=[False, True],
    ).reset_index(drop=True)

    current = {
        split: {
            'rows': fixed_totals[split]['rows'],
            'positives': fixed_totals[split]['positives'],
        }
        for split in SPLIT_ORDER
    }
    moved_rows = 0.0
    assignments = []

    for row in work.itertuples(index=False):
        candidate_splits = list(SPLIT_ORDER)
        rng.shuffle(candidate_splits)
        best_split = None
        best_score = np.inf
        best_moved_rows = None

        for split in candidate_splits:
            projected = {name: values.copy() for name, values in current.items()}
            projected[split]['rows'] += row.row_count
            projected[split]['positives'] += row.positive_count

            group_moved_rows = row.row_count - getattr(row, f'original_{split}_rows')
            projected_moved_rows = moved_rows + group_moved_rows
            score = allocation_objective(projected, projected_moved_rows)

            if score < best_score:
                best_split = split
                best_score = score
                best_moved_rows = projected_moved_rows

        assignments.append(best_split)
        current[best_split]['rows'] += row.row_count
        current[best_split]['positives'] += row.positive_count
        moved_rows = best_moved_rows

    work[NEW_SPLIT_COL] = assignments
    final_objective = allocation_objective(current, moved_rows)
    return work.drop(columns=['_tie_breaker']), current, moved_rows, final_objective


best_crossing_assignment = None
best_current_totals = None
best_moved_rows = None
best_objective = np.inf
attempt_results = []

for attempt in range(N_ATTEMPTS):
    attempt_seed = RANDOM_SEED + attempt
    assignment, current_totals, moved_rows, objective = allocate_crossing_pairs(
        crossing_pair_pd, attempt_seed
    )
    attempt_results.append({
        'attempt': attempt + 1,
        'random_seed': attempt_seed,
        'objective': objective,
        'moved_rows': moved_rows,
    })

    if objective < best_objective:
        best_crossing_assignment = assignment
        best_current_totals = current_totals
        best_moved_rows = moved_rows
        best_objective = objective

assert best_crossing_assignment is not None, 'N_ATTEMPTS must be at least 1'
assignment_records = [
    (str(login_id), str(acct_nbr), str(split))
    for login_id, acct_nbr, split in best_crossing_assignment[
        [EMPLOYEE_COL, ACCOUNT_COL, NEW_SPLIT_COL]
    ].itertuples(index=False, name=None)
]
crossing_assignment_spk = spark.createDataFrame(
    assignment_records,
    schema=f'{EMPLOYEE_COL} string, {ACCOUNT_COL} string, {NEW_SPLIT_COL} string',
)
pair_assignment_spk = (
    non_crossing_assignment_spk
    .unionByName(crossing_assignment_spk)
    .cache()
)

attempt_results_pd = pd.DataFrame(attempt_results).sort_values('objective')
display(attempt_results_pd)
print('Best objective:', best_objective)
print('Moved rows:', best_moved_rows)
print('Final totals:', best_current_totals)
display(pair_assignment_spk.groupBy(NEW_SPLIT_COL).count())

## 9. Join the new split back to every original row

The assignment table has one row per pair.

After joining it back, every original observation receives the split assigned to its employee-account pair.


In [ ]:
features_with_new_split = (
    features_df
    .join(
        pair_assignment_spk,
        on=[EMPLOYEE_COL, ACCOUNT_COL],
        how="left"
    )
)

display(features_with_new_split.limit(10))

## 10. Hard-constraint validation

The following checks must pass:

1. no pair is missing a new split;
2. each pair appears in exactly one new split;
3. no employee-account pair crosses train / val / test;
4. every non-crossing pair remains in its original split;
5. the recorded moved-row count equals the actual number of changed rows.

A feasible result must have:

```text
missing assignments = 0
pairs crossing new splits = 0
non-crossing rows changed = 0
actual moved rows = recorded moved rows
```


In [ ]:
missing_assignment_count = (
    features_with_new_split
    .filter(F.col(NEW_SPLIT_COL).isNull())
    .count()
)

new_pair_integrity = (
    features_with_new_split
    .groupBy(EMPLOYEE_COL, ACCOUNT_COL)
    .agg(
        F.countDistinct(
            NEW_SPLIT_COL
        ).alias("number_of_new_splits")
    )
)

pairs_crossing_new_splits = (
    new_pair_integrity
    .filter(F.col("number_of_new_splits") > 1)
    .count()
)

actual_moved_rows = features_with_new_split.filter(
    F.col(ORIGINAL_SPLIT_COL) != F.col(NEW_SPLIT_COL)
).count()
non_crossing_rows_changed = (
    features_with_new_split
    .join(crossing_pair_ids, on=[EMPLOYEE_COL, ACCOUNT_COL], how='left_anti')
    .filter(F.col(ORIGINAL_SPLIT_COL) != F.col(NEW_SPLIT_COL))
    .count()
)

hard_constraint_summary = pd.DataFrame([{
    "missing_assignments": missing_assignment_count,
    "pairs_crossing_new_splits": pairs_crossing_new_splits,
    'non_crossing_rows_changed': non_crossing_rows_changed,
    'actual_moved_rows': actual_moved_rows,
    "hard_constraint_passed": (
        missing_assignment_count == 0
        and pairs_crossing_new_splits == 0
        and non_crossing_rows_changed == 0
        and actual_moved_rows == int(best_moved_rows)
    )
}])

display(hard_constraint_summary)
assert bool(hard_constraint_summary.loc[0, 'hard_constraint_passed']), (
    'Issue 3A employee-account pair constraint failed'
)

## 11. Compare the original and new split distributions

This is the main feasibility output.

The new allocation is feasible when:

1. pair integrity is fully satisfied;
2. row shares remain close to the original;
3. positive shares remain close to the original;
4. positive rates remain close to the original;
5. validation and test retain enough positive observations for stable evaluation.


In [ ]:
new_split_summary = (
    features_with_new_split
    .groupBy(NEW_SPLIT_COL)
    .agg(
        F.count("*").alias("new_row_count"),
        F.sum(TARGET_COL).alias("new_positive_count"),
        (
            F.count("*") - F.sum(TARGET_COL)
        ).alias("new_negative_count"),
        F.avg(TARGET_COL).alias("new_positive_rate"),
        F.countDistinct(
            F.struct(EMPLOYEE_COL, ACCOUNT_COL)
        ).alias("new_pair_count")
    )
    .withColumn(
        "new_row_share",
        F.col("new_row_count") / F.lit(total_rows)
    )
    .withColumn(
        "new_positive_share",
        F.col("new_positive_count") / F.lit(total_positives)
    )
)

comparison_summary = (
    original_split_summary
    .select(
        F.col(ORIGINAL_SPLIT_COL).alias("split"),
        F.col("row_count").alias("original_row_count"),
        F.col("row_share").alias("original_row_share"),
        F.col("positive_count").alias("original_positive_count"),
        F.col("positive_share").alias("original_positive_share"),
        F.col("positive_rate").alias("original_positive_rate"),
        F.col("pair_count").alias("original_pair_count")
    )
    .join(
        new_split_summary.select(
            F.col(NEW_SPLIT_COL).alias("split"),
            "new_row_count",
            "new_row_share",
            "new_positive_count",
            "new_positive_share",
            "new_positive_rate",
            "new_pair_count"
        ),
        on="split",
        how="inner"
    )
    .withColumn(
        "row_share_difference",
        F.col("new_row_share")
        - F.col("original_row_share")
    )
    .withColumn(
        "positive_share_difference",
        F.col("new_positive_share")
        - F.col("original_positive_share")
    )
    .withColumn(
        "positive_rate_difference",
        F.col("new_positive_rate")
        - F.col("original_positive_rate")
    )
    .orderBy(
        F.when(F.col("split") == "train", 1)
         .when(F.col("split") == "val", 2)
         .when(F.col("split") == "test", 3)
         .otherwise(4)
    )
)

display(comparison_summary)

## 12. Simple feasibility criteria

These thresholds are validation guidelines rather than universal statistical laws.

Suggested criteria:

- pair integrity: exactly zero crossing pairs;
- absolute row-share difference: no more than 2 percentage points;
- absolute positive-share difference: no more than 2 percentage points;
- positive-rate relative difference: no more than 10%;
- validation and test must each contain at least a configurable minimum number of positives.

The minimum-positive threshold should be selected according to the project's evaluation requirements.


In [ ]:
# Configurable review thresholds
MAX_ABSOLUTE_ROW_SHARE_DIFFERENCE = 0.02
MAX_ABSOLUTE_POSITIVE_SHARE_DIFFERENCE = 0.02
MAX_RELATIVE_POSITIVE_RATE_DIFFERENCE = 0.10
MIN_POSITIVES_IN_VAL_OR_TEST = 100

comparison_pd = comparison_summary.toPandas()

comparison_pd["absolute_row_share_difference"] = (
    comparison_pd["row_share_difference"].abs()
)

comparison_pd["absolute_positive_share_difference"] = (
    comparison_pd["positive_share_difference"].abs()
)

comparison_pd["relative_positive_rate_difference"] = (
    (
        comparison_pd["new_positive_rate"]
        - comparison_pd["original_positive_rate"]
    ).abs()
    / comparison_pd["original_positive_rate"].replace(0, np.nan)
)

comparison_pd["row_balance_passed"] = (
    comparison_pd["absolute_row_share_difference"]
    <= MAX_ABSOLUTE_ROW_SHARE_DIFFERENCE
)

comparison_pd["positive_share_balance_passed"] = (
    comparison_pd["absolute_positive_share_difference"]
    <= MAX_ABSOLUTE_POSITIVE_SHARE_DIFFERENCE
)

comparison_pd["positive_rate_balance_passed"] = (
    comparison_pd["relative_positive_rate_difference"]
    <= MAX_RELATIVE_POSITIVE_RATE_DIFFERENCE
)

comparison_pd["minimum_positive_count_passed"] = True

comparison_pd.loc[
    comparison_pd["split"].isin(["val", "test"]),
    "minimum_positive_count_passed"
] = (
    comparison_pd.loc[
        comparison_pd["split"].isin(["val", "test"]),
        "new_positive_count"
    ]
    >= MIN_POSITIVES_IN_VAL_OR_TEST
)

all_balance_checks_passed = bool(
    comparison_pd[
        [
            "row_balance_passed",
            "positive_share_balance_passed",
            "positive_rate_balance_passed",
            "minimum_positive_count_passed"
        ]
    ]
    .all()
    .all()
)

hard_constraint_passed = bool(
    hard_constraint_summary.loc[
        0,
        "hard_constraint_passed"
    ]
)

overall_feasible = (
    hard_constraint_passed
    and all_balance_checks_passed
)

display(comparison_pd)

print("Hard constraint passed:", hard_constraint_passed)
print("All balance checks passed:", all_balance_checks_passed)
print("Overall split considered feasible:", overall_feasible)

## 13. Optional diagnostic: largest crossing pairs

Large crossing employee-account pairs can make perfect balancing impossible because they cannot be divided.

This table identifies pairs that have the greatest influence on row and positive-label allocation.


In [ ]:
largest_pairs = (
    crossing_pair_summary_spk
    .orderBy(
        F.desc("row_count"),
        F.desc("positive_count")
    )
)

display(largest_pairs.limit(50))

## 14. Build and save the complete feature table

The original full feature table is read again and joined to the repaired split by its sample keys. The saved output keeps the original schema and column order, replaces `label_split`, and rebuilds `cv_fold` for the new training rows.

Training rows receive a label-stratified random five-fold assignment. Validation and test rows receive a null `cv_fold`. The original feature table is not overwritten.


In [ ]:
assert overall_feasible, 'The repaired split did not pass validation'

full_feature_source = (
    spark.read
    .format('delta')
    .load(FEATURE_TABLE_PATH)
    .cache()
)
source_row_count = full_feature_source.count()
assert source_row_count == total_rows, (
    f'Full feature source has {source_row_count} rows; split analysis has {total_rows}'
)

mapping_keys = [
    EMPLOYEE_COL, ACCOUNT_COL, SAMPLE_DATE_COL,
    WINDOW_START_COL, WINDOW_END_COL,
]
split_mapping = (
    features_with_new_split
    .select(*mapping_keys, NEW_SPLIT_COL)
    .dropDuplicates(mapping_keys)
    .cache()
)
mapping_row_count = split_mapping.count()
assert mapping_row_count == total_rows, (
    f'Split mapping has {mapping_row_count} unique keys; expected {total_rows}'
)

source_alias = full_feature_source.alias('source')
mapping_alias = split_mapping.alias('mapping')
join_conditions = [
    F.upper(F.col(f'source.{EMPLOYEE_COL}')) == F.col(f'mapping.{EMPLOYEE_COL}'),
    F.col(f'source.{ACCOUNT_COL}').cast('string') == F.col(f'mapping.{ACCOUNT_COL}'),
    F.to_date(F.col(f'source.{SAMPLE_DATE_COL}')) == F.col(f'mapping.{SAMPLE_DATE_COL}'),
    F.to_date(F.col(f'source.{WINDOW_START_COL}')) == F.col(f'mapping.{WINDOW_START_COL}'),
    F.to_date(F.col(f'source.{WINDOW_END_COL}')) == F.col(f'mapping.{WINDOW_END_COL}'),
]

columns_to_keep = [
    column for column in full_feature_source.columns
    if column not in [ORIGINAL_SPLIT_COL, 'cv_fold']
]
complete_features_base = (
    source_alias
    .join(mapping_alias, on=join_conditions, how='left')
    .select(
        *[F.col(f'source.`{column}`').alias(column) for column in columns_to_keep],
        F.col(f'mapping.{NEW_SPLIT_COL}').alias(ORIGINAL_SPLIT_COL),
    )
    .cache()
)

joined_row_count = complete_features_base.count()
missing_split_count = complete_features_base.filter(
    F.col(ORIGINAL_SPLIT_COL).isNull()
).count()
assert joined_row_count == total_rows, f'Join produced {joined_row_count}; expected {total_rows}'
assert missing_split_count == 0, f'{missing_split_count} rows did not receive the repaired split'

train_fold_window = (
    Window.partitionBy(TARGET_COL)
    .orderBy(F.rand(seed=CV_FOLD_SEED))
)
new_training_rows = (
    complete_features_base
    .filter(F.col(ORIGINAL_SPLIT_COL) == 'train')
    .withColumn('_fold_rank', F.row_number().over(train_fold_window))
    .withColumn(
        'cv_fold',
        ((F.col('_fold_rank') - 1) % N_CV_FOLDS).cast('int'),
    )
    .drop('_fold_rank')
)
new_validation_and_test_rows = (
    complete_features_base
    .filter(F.col(ORIGINAL_SPLIT_COL) != 'train')
    .withColumn('cv_fold', F.lit(None).cast('int'))
)

final_feature_table = (
    new_training_rows
    .unionByName(new_validation_and_test_rows)
    .select(*full_feature_source.columns)
    .cache()
)

invalid_cv_fold_count = final_feature_table.filter(
    (
        (F.col(ORIGINAL_SPLIT_COL) == 'train')
        & (F.col('cv_fold').isNull() | ~F.col('cv_fold').between(0, N_CV_FOLDS - 1))
    )
    | (
        (F.col(ORIGINAL_SPLIT_COL) != 'train')
        & F.col('cv_fold').isNotNull()
    )
).count()
assert final_feature_table.count() == source_row_count
assert final_feature_table.columns == full_feature_source.columns
assert invalid_cv_fold_count == 0, f'Found {invalid_cv_fold_count} rows with invalid cv_fold'

if 'insider_label' in final_feature_table.columns:
    insiders_outside_test = final_feature_table.filter(
        (F.col('insider_label') == 1)
        & (F.col(ORIGINAL_SPLIT_COL) != 'test')
    ).count()
    assert insiders_outside_test == 0, (
        f'{insiders_outside_test} insider_label rows are outside test'
    )

display(
    final_feature_table
    .groupBy(ORIGINAL_SPLIT_COL, 'cv_fold', TARGET_COL)
    .count()
    .orderBy(ORIGINAL_SPLIT_COL, 'cv_fold', TARGET_COL)
)

(
    final_feature_table.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(FULL_FEATURE_OUTPUT_PATH)
)
print('Saved complete feature table:', FULL_FEATURE_OUTPUT_PATH)

## Interpretation

The result supports a **Yes** answer to issue A when:

1. every employee-account pair is assigned to exactly one split;
2. the new row allocation remains close to the original MD allocation;
3. positive counts and positive rates remain acceptably balanced;
4. validation and test retain enough positive observations;
5. every non-crossing pair remains in its original split, while crossing pairs are repaired with the lowest observed global loss.

After the complete feature table is saved, the cloned LightGBM notebook can read the new path directly. Its `label_split` and `cv_fold` columns already contain the repaired values needed for a fair in-time and OOT comparison.
